# Etapa 1. Conferência e consolidação

**Entrada:** os arquivos anuais de `00_extracao/dados_brutos` (2016 a 2025), extraídos do Tesouro Gerencial com `00_extracao/sql/extracao_mestrado_v2.sql`.

**Processo:**
1. Conferência de qualidade de todos os arquivos (`src/qualidade.py`).
2. Consolidação numa base única, com os saldos mensais (`src/consolidacao.py`), que só roda se a conferência não tiver FALHA.

**Saída** (nesta pasta):
- `relatorio_qualidade.xlsx`: resultado de todas as conferências.
- `base_consolidada/`: `movimentos.parquet`, `saldos_mensais.parquet`, `dimensoes.xlsx` e `LEIAME.md`.

Status das conferências: **OK** (passou), **ATENÇÃO** (diferença esperada ou a examinar, que não impede seguir) e **FALHA** (problema a resolver antes da consolidação). As regras de inclusão do universo (UFNT fora, 2020 fora para as universidades novas) não são aplicadas aqui: a base é completa, e as regras entram na montagem de cada desenho.

## Preparação

In [4]:
import os
import sys
from pathlib import Path

import pandas as pd

# O notebook roda dentro da pasta da etapa. Aqui ele localiza a raiz do repositório
# (a pasta que contém src/) e passa a trabalhar a partir dela.
raiz = Path.cwd()
while not (raiz / "src").is_dir():
    if raiz.parent == raiz:
        raise FileNotFoundError("Pasta src não encontrada. Abra o notebook dentro do repositório.")
    raiz = raiz.parent
os.chdir(raiz)
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.qualidade import conferir
from src.consolidacao import consolidar

PASTA_BRUTOS = Path("00_extracao/dados_brutos")
PASTA_ETAPA = Path("01_conferencia_e_consolidacao")
PASTA_BASE = PASTA_ETAPA / "base_consolidada"
RELATORIO = PASTA_ETAPA / "relatorio_qualidade.xlsx"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print(f"Raiz do repositório: {raiz}")
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")

Raiz do repositório: c:\Users\silvar\Downloads\pipeline_projeto_mestrado-versao-3\pipeline_projeto_mestrado-versao-3
Python 3.13.14 | pandas 2.3.3


## 1. Conferência de qualidade

In [5]:
resumo = conferir(PASTA_BRUTOS, PASTA_ETAPA)
resumo

Arquivos encontrados: exportacao_2016.csv, exportacao_2017.csv, exportacao_2018.csv, exportacao_2019.csv, exportacao_2020.csv, exportacao_2021.csv, exportacao_2022.csv, exportacao_2023.csv, exportacao_2024.csv, exportacao_2025.csv

Lendo exportacao_2016.csv ... 572.194 linhas | status do arquivo: OK
Lendo exportacao_2017.csv ... 577.431 linhas | status do arquivo: OK
Lendo exportacao_2018.csv ... 608.698 linhas | status do arquivo: OK
Lendo exportacao_2019.csv ... 617.860 linhas | status do arquivo: OK
Lendo exportacao_2020.csv ... 547.094 linhas | status do arquivo: OK
Lendo exportacao_2021.csv ... 528.545 linhas | status do arquivo: OK
Lendo exportacao_2022.csv ... 560.540 linhas | status do arquivo: OK
Lendo exportacao_2023.csv ... 572.250 linhas | status do arquivo: OK
Lendo exportacao_2024.csv ... 578.190 linhas | status do arquivo: OK
Lendo exportacao_2025.csv ... 584.011 linhas | status do arquivo: OK

RESUMO DA CONFERÊNCIA
[  OK   ] Sequência de anos | com problema: 0
[  OK   ]

,conferencia,status,verificados,com_problema,observacao
0,Sequência de anos,OK,2016 a 2025 (10 arquivos),0,
1,"Estrutura dos arquivos (colunas, ano, meses, u...",OK,10,0,Ver aba Arquivos
2,Igualdades contábeis por universidade e ano (a...,ATENÇÃO,426,6,Ver aba Universidade_ano
3,Continuidade entre anos: total de Ativo e de P...,OK,762,0,Fechamento de um ano contra abertura do seguin...
4,Continuidade entre anos: grupos de contas (3 d...,ATENÇÃO,4774,20,Diferenças com total igual indicam reclassific...
5,Continuidade entre anos: contas individuais,ATENÇÃO,,832,Informativo. Ver aba Continuidade_contas
6,Confronto com os diagnósticos SQL (UFF),OK,100,0,Valores calculados sem a tabela de câmbio


### Detalhes do que não passou

As células abaixo mostram só as linhas com ATENÇÃO ou FALHA de cada aba do relatório. Tabelas vazias significam que não houve problema naquela conferência.

In [6]:
abas = pd.read_excel(RELATORIO, sheet_name=None)

print("Arquivos com status diferente de OK:")
arquivos = abas["Arquivos"]
display(arquivos[arquivos["status"] != "OK"])

print("Universidades e anos com status diferente de OK:")
univ = abas["Universidade_ano"]
display(univ[univ["status"] != "OK"])

Arquivos com status diferente de OK:


,ano,arquivo,linhas,colunas_necessarias_faltando,cabecalho_igual_ao_primeiro_ano,anos_no_arquivo,meses_presentes,meses_fora_de_0_a_14,universidades,universidades_faltando,universidades_inesperadas,valores_nao_numericos,contas_fora_do_padrao_9_digitos,classes_inesperadas,linhas_duplicadas,linhas_mes_14,status


Universidades e anos com status diferente de OK:


,ano,cod_orgao,tem_mes_0,tem_mes_13,meses_1_a_12_com_movimento,primeiro_mes_com_movimento,abertura_ativo_menos_passivo_pl,fechamento_ate_mes_13,fechamento_final,vpd_acumulada_fim,vpa_acumulada_fim,status
195,2020,26452,False,True,12,1,NaN,0,0,0,0,ATENÇÃO
196,2020,26453,False,True,12,1,NaN,0,0,0,0,ATENÇÃO
197,2020,26454,False,True,12,1,NaN,0,0,0,0,ATENÇÃO
198,2020,26455,False,True,12,1,NaN,0,0,0,0,ATENÇÃO
199,2020,26456,False,True,12,1,NaN,0,0,0,0,ATENÇÃO
200,2020,26457,False,True,2,11,NaN,0,0,0,0,ATENÇÃO


In [7]:
TOLERANCIA = 0.05
grupos = abas["Continuidade_grupos"]
if len(grupos):
    print("Continuidade entre anos, por classe (1 = Ativo, 2 = Passivo e PL), com diferença:")
    classe = grupos[grupos["nivel"] == "classe"]
    display(classe[classe["diferenca"].abs() > TOLERANCIA])
    print("Continuidade entre anos, por grupo de 3 dígitos, com diferença:")
    grupo = grupos[grupos["nivel"] == "grupo"]
    display(grupo[grupo["diferenca"].abs() > TOLERANCIA])
    print(f"Contas individuais com diferença: {len(abas['Continuidade_contas'])} (ver aba Continuidade_contas)")
else:
    print("Só há um ano: a continuidade entre anos não se aplica.")

Continuidade entre anos, por classe (1 = Ativo, 2 = Passivo e PL), com diferença:


,ano,cod_orgao,nivel,codigo,fechamento_ano,abertura_ano_seguinte,diferenca


Continuidade entre anos, por grupo de 3 dígitos, com diferença:


,ano,cod_orgao,nivel,codigo,fechamento_ano,abertura_ano_seguinte,diferenca
2439,2020,26239,grupo,215,0.00,"3,760.00","3,760.00"
2440,2020,26239,grupo,218,"228,432,544.38","228,428,784.38","-3,760.00"
2470,2020,26241,grupo,215,0.00,"1,675,135.14","1,675,135.14"
2471,2020,26241,grupo,218,"289,250,173.65","287,575,038.51","-1,675,135.14"
2485,2020,26242,grupo,215,0.00,"1,200.00","1,200.00"
2486,2020,26242,grupo,218,"250,715,858.81","250,714,658.81","-1,200.00"
2514,2020,26244,grupo,215,0.00,"1,982,117.82","1,982,117.82"
2515,2020,26244,grupo,218,"250,198,280.50","248,216,162.68","-1,982,117.82"
2527,2020,26245,grupo,215,0.00,"84,506.66","84,506.66"
2528,2020,26245,grupo,218,"468,334,820.44","468,250,313.78","-84,506.66"


Contas individuais com diferença: 832 (ver aba Continuidade_contas)


In [8]:
print("Confronto com os diagnósticos SQL da UFF (diferenças acima da tolerância):")
diag = abas["Diagnosticos"]
display(diag[diag["diferenca"].abs() > TOLERANCIA] if len(diag) else diag)

Confronto com os diagnósticos SQL da UFF (diferenças acima da tolerância):


,ano,conferencia,extracao,diagnostico,diferenca


### Universidades novas e volume por universidade

Primeiro mês com movimento das 6 universidades criadas a partir de 2018, para confirmar a exclusão de 2020 (D28) e a situação da UFNT (D29), e número de linhas por universidade e ano.

In [9]:
display(abas["Novas_universidades"])
display(abas["Linhas_por_universidade"])

,ano,cod_orgao,tem_mes_0,meses_1_a_12_com_movimento,primeiro_mes_com_movimento
0,2020,26452,False,12,1
1,2020,26453,False,12,1
2,2020,26454,False,12,1
3,2020,26455,False,12,1
4,2020,26456,False,12,1
5,2020,26457,False,2,11
6,2021,26452,True,12,1
7,2021,26453,True,12,1
8,2021,26454,True,12,1
9,2021,26455,True,12,1


,cod_orgao,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,26231,11249,11438,12663,12692,11507,10729,11142,11466,11141,11274
1,26232,16137,16650,17984,18206,16160,14757,14847,14848,15150,15086
2,26233,15391,16075,17212,17299,15405,14349,14427,14386,14893,14664
3,26234,16037,15706,16786,17037,14900,13700,14566,14609,14904,15132
4,26235,13181,13130,13876,19073,13055,11701,12094,12085,12156,12145
5,26236,20979,21237,22045,22778,19573,18691,19534,19018,19456,19058
6,26237,11436,11021,11986,12538,11831,11197,11666,11976,12277,12644
7,26238,63946,64551,66945,66233,50534,48232,53043,55416,54878,56542
8,26239,15947,16103,16851,17712,15459,14073,14104,14336,14181,13802
9,26240,18573,18406,19347,18854,16272,15666,15335,13523,14500,14270


## 2. Consolidação

Só roda se o relatório de qualidade não tiver FALHA. Se houver, a célula para com uma mensagem, e a falha deve ser investigada antes.

In [10]:
consolidar(PASTA_BRUTOS, PASTA_BASE, PASTA_ETAPA)

Relatório de qualidade sem falhas. Iniciando a consolidação.

Lendo exportacao_2016.csv ... 415.621 linhas com movimento
Lendo exportacao_2017.csv ... 417.545 linhas com movimento
Lendo exportacao_2018.csv ... 443.659 linhas com movimento
Lendo exportacao_2019.csv ... 484.170 linhas com movimento
Lendo exportacao_2020.csv ... 422.136 linhas com movimento
Lendo exportacao_2021.csv ... 405.799 linhas com movimento
Lendo exportacao_2022.csv ... 432.686 linhas com movimento
Lendo exportacao_2023.csv ... 442.738 linhas com movimento
Lendo exportacao_2024.csv ... 453.574 linhas com movimento
Lendo exportacao_2025.csv ... 460.206 linhas com movimento

Base consolidada:
  movimentos.parquet               44.0 MB
  saldos_mensais.parquet            7.6 MB
  dimensoes.xlsx                    0.0 MB
  movimentos: 4.378.134 linhas | saldos: 1.660.044 linhas
  anos: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] | universidades: 45

Arquivos gravados em C:\Users\silvar\Downloads\pipel

### Conferência rápida da base consolidada

In [11]:
mov = pd.read_parquet(PASTA_BASE / "movimentos.parquet")
sal = pd.read_parquet(PASTA_BASE / "saldos_mensais.parquet")

print(f"movimentos: {len(mov):,} linhas | saldos: {len(sal):,} linhas".replace(",", "."))
print("Universidades por ano:")
display(mov.groupby("ano")["cod_orgao"].nunique().rename("universidades").to_frame().T)

# Ativo = Passivo + PL no fim de cada exercício (saldo do mês 13), para todas as universidades e anos
fim = sal[sal["mes"] == 13].assign(classe=lambda d: d["cod_conta"].astype(str).str[0])
fim = fim[fim["classe"].isin(["1", "2"])].pivot_table(index=["ano", "cod_orgao"], columns="classe",
                                                      values="saldo", aggfunc="sum", fill_value=0.0)
dif = (fim["1"] - fim["2"]).abs()
print(f"Ativo menos Passivo + PL no fim do exercício: maior diferença = R$ {dif.max():,.2f} "
      f"em {len(dif)} universidades-ano")

movimentos: 4.378.134 linhas | saldos: 1.660.044 linhas
Universidades por ano:


ano,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
universidades,39,39,39,39,45,45,45,45,45,45


Ativo menos Passivo + PL no fim do exercício: maior diferença = R$ 0.00 em 426 universidades-ano


## Registro

Preencher depois de rodar, para o diário de decisões:

- Data da execução: 13/11/2026
- Resultado da conferência: Tudo ok
- Observações sobre ATENÇÃO ou FALHA: pouquíssimas "ATENÇÃO", e não foram nada relevante
- Tamanho da base consolidada: 44.0 + 7.6 mb